In [5]:
import glob
import json
import yaml
import shutil
import numpy as np
import pandas as pd
import datetime
from ray import tune
import hashlib
import rampwf as rw
import ramphy as rh
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi
#import altair as alt
#alt.renderers.enable('default')
pd.options.display.max_columns = 200
pd.options.display.max_rows = 1000
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import DatetimeTickFormatter, Label
from bokeh.resources import CDN
from bokeh.embed import file_html
output_notebook()

kits_root = Path("/home/gpaolo/data/ramp-kits")

/tmp/ipykernel_3817424/2606921247.py:18: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


Loading BokehJS ...

In [6]:
def kaggle_select(kaggle_api, suffix, id):
    f_name = kaggle_api.string(getattr(id, "fileName"))
    select = f_name[5:5 + len(suffix)] == suffix
    select = select and kaggle_api.string(getattr(id, "publicScore")) != ''
    return select

def ramp_kit_plot(ramp_kit_dir):
    problem = rw.utils.assert_read_problem(ramp_kit_dir=ramp_kit_dir)
    score_names = [st.name for st in problem.score_types]
    score_type = problem.score_types[-1]
    valid_score_name = f'valid_{score_names[-1]}'
    metadata = json.load(open(Path(ramp_kit_dir) / "data" / "metadata.json"))
    kaggle_api = KaggleApi()
    kaggle_api.authenticate()
    
    action_f_names = glob.glob(f'{ramp_kit_dir}/actions/*')
    action_f_names.sort()
    ramp_program = []
    start_time = datetime.datetime(2024, 5, 8, 12, 30, 00)
    for action_f_name in action_f_names:
        f_name = Path(action_f_name).name
        if f_name > str(start_time):
            ramp_program.append(rh.actions.load_ramp_action(action_f_name))
    
    hyperopt_actions = [ra for ra in ramp_program if ra.start_time > start_time and ra.name == "hyperopt"]
    blend_actions = [ra for ra in ramp_program if ra.start_time > start_time and (ra.name == "blend" or ra.name == "bag_then_blend")]
    train_actions = [ra for ra in ramp_program if ra.start_time > start_time and ra.name == "train"]
    print(blend_actions[-1].__dict__)
    start_time = min([ra.start_time for ra in ramp_program if ra.name == "hyperopt"])
    stop_time = max([ra.stop_time for ra in ramp_program])
    print(f"{len(hyperopt_actions)} rounds done")
    
    color_dict = {
        "catboost": "#2ca02c",
        "lgbm": "#ff7f0e",
        "xgboost": "#9467bd",
        "transformer": "#1f77b4",
    }
    tooltips = [
        ("submission", "@submission"),
        ("score", "@valid_score"),
        ("run time [s]", "@runtime"),
    ]
    p = figure(
        title=f"AutoDS on {ramp_kit_dir}", x_axis_label='action time',
        x_axis_type="datetime", y_axis_label=f"valid {score_names[-1]}",
        tooltips=tooltips, width=1000, height=500,
    )
    ### Kaggle private leaderboard percentile rank lines
    quantile_start = 0.5
    quantile_resolution = 0.05
    try:
        public_leaderboard_scores = np.load(Path(ramp_kit_dir) / "data" / "public_leaderboard_scores.npy")
    except:
        pass
    try:
        private_leaderboard_scores = np.load(Path(ramp_kit_dir) / "data" / "private_leaderboard_scores.npy")
        if score_type.is_lower_the_better:
            private_leaderboard_ranks = np.clip(np.arange(0, quantile_start + 0.0001, quantile_resolution), 0, 1)
        else:
            private_leaderboard_ranks = np.clip(np.arange(quantile_start, 1.01, quantile_resolution), 0, 1)
        private_leaderboard_quantiles = np.quantile(private_leaderboard_scores, private_leaderboard_ranks)
        for qi, quantile in enumerate(private_leaderboard_quantiles):
            if score_type.is_lower_the_better:
                percentile_rank = int((1 - private_leaderboard_ranks[qi]) * 100)
            else:
                percentile_rank = int(private_leaderboard_ranks[qi] * 100)
            source = pd.DataFrame({
                "submission": [percentile_rank, percentile_rank],
                "time": [start_time, stop_time],
                "valid_score": [quantile, quantile],
            })
            p.line("time", "valid_score", line_width=1, color="#d62728", alpha=0.5, source=source)
            rank_text = Label(
                x=stop_time, y=quantile, text=str(percentile_rank), text_color="#d62728", text_alpha=0.5,
                text_align="left", text_font_size="6px", text_baseline="middle")
            p.add_layout(rank_text)     
    except:
        try:
            if score_type.is_lower_the_better:
                public_leaderboard_ranks = np.clip(np.arange(0, quantile_start + 0.0001, quantile_resolution), 0, 1)
            else:
                public_leaderboard_ranks = np.clip(np.arange(quantile_start, 1.01, quantile_resolution), 0, 1)
            public_leaderboard_quantiles = np.quantile(public_leaderboard_scores, public_leaderboard_ranks)
            for qi, quantile in enumerate(public_leaderboard_quantiles):
                if score_type.is_lower_the_better:
                    percentile_rank = int((1 - public_leaderboard_ranks[qi]) * 100)
                else:
                    percentile_rank = int(public_leaderboard_ranks[qi] * 100)
                source = pd.DataFrame({
                    "submission": [percentile_rank, percentile_rank],
                    "time": [start_time, stop_time],
                    "valid_score": [quantile, quantile],
                })
                p.line("time", "valid_score", line_width=1, color="#1f77b4", alpha=0.5, source=source)
                rank_text = Label(
                    x=stop_time, y=quantile, text=str(percentile_rank), text_color="#1f77b4", text_alpha=0.5,
                    text_align="left", text_font_size="6px", text_baseline="middle")
                p.add_layout(rank_text)     
        except:
            pass
    
    ### Hyperopt race action segment
    for ra in hyperopt_actions:
        if len(ra.mean_scores) > 0:
            submission = ra.kwargs["submission"]
            source = pd.DataFrame({
                "action time": [ra.start_time, ra.stop_time],
                "valid_score": [ra.mean_score, ra.mean_score],
                "submission": [submission, submission],
                "runtime": [int(ra.runtime.total_seconds()), int(ra.runtime.total_seconds())],
            })
            p.line("action time", "valid_score", legend_label=submission, line_width=3, color=color_dict[submission], source=source)
    
    ### Blended score after each action
    source = pd.DataFrame({
        "action time": [ra.start_time for ra in blend_actions],
        "valid_score": [ra.blended_score for ra in blend_actions],
        "submission": [ra.kwargs["submissions"] for ra in blend_actions],
        "runtime": [ra.runtime.total_seconds() for ra in blend_actions],
    })
    p.line("action time", "valid_score", legend_label='blend', line_width=3, color="black", source=source)
    
    ### Kaggle public and private scores
    try:            
        kaggle_submission_ids = kaggle_api.competition_submissions(competition=metadata["kaggle_name"])
        kaggle_submission_ids = [id for id in kaggle_submission_ids
                                 if not "_best_" in kaggle_api.string(getattr(id, "fileName"))]
        kaggle_public_scores = [kaggle_api.string(getattr(id, "publicScore")) for id in kaggle_submission_ids
                                if kaggle_select(kaggle_api, kit_suffix, id)]
        kaggle_private_scores = [kaggle_api.string(getattr(id, "privateScore")) for id in kaggle_submission_ids
                                 if kaggle_select(kaggle_api, kit_suffix, id)]
        kaggle_action_times = [kaggle_api.string(getattr(id, "description")) for id in kaggle_submission_ids
                               if kaggle_select(kaggle_api, kit_suffix, id)]
        kaggle_file_names = [kaggle_api.string(getattr(id, "fileName")) for id in kaggle_submission_ids
                             if kaggle_select(kaggle_api, kit_suffix, id)]
        source = pd.DataFrame({
            "action time": kaggle_action_times,
            "valid_score": kaggle_public_scores,
            "submission": kaggle_file_names,
        })
        source["action time"] = pd.to_datetime(source["action time"], format='mixed')
        p.line("action time", "valid_score", legend_label='public kaggle', line_width=3, color="#1f77b4", source=source)
        
        if "".join(kaggle_private_scores) != "":
            source = pd.DataFrame({
                "action time": kaggle_action_times,
                "valid_score": kaggle_private_scores,
                "submission": kaggle_file_names,
            })
            source["action time"] = pd.to_datetime(source["action time"], format='mixed')
            p.line("action time", "valid_score", legend_label='private kaggle', line_width=3, color="#d62728", source=source)
            
            final_time = max(source["action time"])
            final_score = float(source[source["action time"] == final_time]["valid_score"].to_numpy()[0])
            if score_type.is_lower_the_better:
                final_rank = np.less(float(final_score), private_leaderboard_scores).mean()
            else:
                final_rank = np.greater(float(final_score), private_leaderboard_scores).mean()
            final_rank_text = Label(
                x=final_time, y=final_score, text=str(round(100 * final_rank)), text_color="#d62728", text_alpha=1,
                text_align="left", text_font_size="18px", text_baseline="middle")
            p.add_layout(final_rank_text)
        else:
            final_time = max(source["action time"])
            final_score = float(source[source["action time"] == final_time]["valid_score"].to_numpy()[0])
            if score_type.is_lower_the_better:
                final_rank = np.less(float(final_score), public_leaderboard_scores).mean()
            else:
                final_rank = np.greater(float(final_score), public_leaderboard_scores).mean()
            final_rank_text = Label(
                x=final_time, y=final_score, text=str(round(100 * final_rank)), text_color="#1f77b4", text_alpha=1,
                text_align="left", text_font_size="18px", text_baseline="middle")
            p.add_layout(final_rank_text)     
    except Exception as e:
        print(e)
        pass
    
    ### Training actions after the end of the hyperopt race
    for ra in train_actions:
        submission = ra.kwargs["submission"]
        base_submission = submission[:-len("_hyperopt_0000000000")]
        if base_submission in color_dict.keys():
            source = pd.DataFrame({
                "action time": [ra.start_time, ra.stop_time],
                "valid_score": [ra.mean_score, ra.mean_score],
                "submission": [submission, submission],
                "runtime": [int(ra.runtime.total_seconds()), int(ra.runtime.total_seconds())],
            })
            p.line("action time", "valid_score", legend_label=base_submission, line_width=3, color=color_dict[base_submission], source=source)
    
    p.xaxis[0].formatter = DatetimeTickFormatter(hourmin = "%b%d %H:%M", minutes = "%H:%M", minsec = "%H:%M:%Ss")
    return p

In [12]:
ramp_kit = f"kaggle_cars_text"
versions = ["tokenizer"]
number = "X"

for version in versions:
    kit_suffix = f"v{version}_n{number}"
    ramp_kit_dir = kits_root / f"{ramp_kit}_{kit_suffix}"

    p = ramp_kit_plot( ramp_kit_dir)
    show(p)

    plots_dir = Path(ramp_kit_dir) / "plots"
    plots_dir.mkdir(parents=False, exist_ok=True)
    html = file_html(p, CDN, "plot")
    with open(plots_dir / "summary_plot.html", "w") as f:
        f.write(html)

IndexError: list index out of range

In [ ]:
ramp_kit = f"kaggle_synthanic"
version = "1_1"
numbers = ["X", "X_pca", "llama3", "X_llama3"]

for number in numbers:
    kit_suffix = f"v{version}_n{number}"
    ramp_kit_dir = kits_root / f"{ramp_kit}_{kit_suffix}"

    p = ramp_kit_plot( ramp_kit_dir)
    show(p)

    plots_dir = Path(ramp_kit_dir) / "plots"
    plots_dir.mkdir(parents=False, exist_ok=True)
    html = file_html(p, CDN, "plot")
    with open(plots_dir / "summary_plot.html", "w") as f:
        f.write(html)


In [ ]:
results_summary_df = pd.read_csv(kits_root / "results_summary.csv")
for col in results_summary_df.columns:
    if col[:14] == "contributivity" or col[:6] == "rounds":
        results_summary_df[col] = results_summary_df[col].fillna(0)
        results_summary_df[col] = results_summary_df[col].astype("int64")
    if col[:7] == "runtime":
        results_summary_df[col] = results_summary_df[col].astype("timedelta64[ns]")
for row_i, row in results_summary_df.iterrows():
    if row["kaggle_finished"] == 1:
        kit_suffix = f"v{row['version']}_n{row['number']}"
        ramp_kit_dir = f"{row['ramp_kit']}_{kit_suffix}"
        problem = rw.utils.assert_read_problem(ramp_kit_dir=kits_root/ ramp_kit_dir)
        score_type = problem.score_types[-1]
        final_cols = ["last_blend", "bagged_then_blended"]
        eval_col = "kaggle_public"
#        eval_col = "valid"
        eval_cols = [f"{eval_col}_{fc}" for fc in final_cols]
        if score_type.is_lower_the_better:
            best_public_submission = row[eval_cols].idxmin()[len(eval_col) + 1:]
        else:
            best_public_submission = row[eval_cols].idxmax()[len(eval_col) + 1:]
        if np.isnan(row[f"kaggle_private_prank_{best_public_submission}"]):
            results_summary_df.loc[row_i, "kaggle_private_prank_best_public"] = row[f"kaggle_public_prank_{best_public_submission}"]
#            results_summary_df.loc[row_i, "kaggle_private_prank_best_public"] = row[f"kaggle_public_prank_growing_folds"]
        else:
            results_summary_df.loc[row_i, "kaggle_private_prank_best_public"] = row[f"kaggle_private_prank_{best_public_submission}"]
#            results_summary_df.loc[row_i, "kaggle_private_prank_best_public"] = row[f"kaggle_private_prank_growing_folds"]
        results_summary_df.loc[row_i, "kaggle_private_prank_best_submission"] = best_public_submission
#results_summary_df["number"] = results_summary_df["number"].astype(int)
#results_summary_df["run_finished"] = 0
#results_summary_df["kaggle_finished"] = 0
results_summary_df

In [ ]:
public_summary_df = results_summary_df[(results_summary_df["run_finished"] == 1) & (results_summary_df["kaggle_finished"] == 1)].drop(
    columns=["server", "kaggle_private_prank_best_submission", "version"]
    ).groupby(["ramp_kit", "number"]).mean()
public_summary_df["AutoDS"] = round(public_summary_df["kaggle_private_prank_best_public"])
public_summary_df["AutoDS"] = round(public_summary_df["AutoDS"].fillna(public_summary_df["kaggle_private_prank_best_public"]))
public_summary_df = public_summary_df[["AutoDS"]].astype(int).reset_index()
public_summary_df["Kaggle challenge"] = ""
for row_i, row in public_summary_df.iterrows():
    kit_suffix = f"v1_1_n{row['number']}"
    ramp_kit_dir = f"{row['ramp_kit']}_{kit_suffix}"
    metadata = json.load(open(kits_root / ramp_kit_dir / "data" / "metadata.json"))
    public_summary_df.loc[row_i, "Kaggle challenge"] = f"{metadata['title']} {metadata['prediction_type']}"  
public_summary_df = public_summary_df.set_index("ramp_kit")
weco_df = pd.read_csv(kits_root / "weco.csv").set_index("ramp_kit")
public_summary_df = public_summary_df.join(weco_df)
public_summary_df = public_summary_df[["Kaggle challenge", "number", "AutoDS", "kaggle_prank"]]
public_summary_df = public_summary_df.rename(columns={
    "Kaggle challenge": "Kaggle challenge", "number": "AutoDs version", "AutoDS": "AutoDS % rank", "kaggle_prank": "WecoAI % rank"})

In [ ]:
public_summary_df

In [ ]:
results_summary_df["contributivity_bagged_then_blended_lgbm"] = np.nan
results_summary_df["contributivity_bagged_then_blended_xgboost"] = np.nan
results_summary_df["contributivity_bagged_then_blended_catboost"] = np.nan
cols = list(results_summary_df.columns)
results_summary_df = results_summary_df[cols[0:42] + cols[-3:] + cols[42:-3]]

In [ ]:
results_summary_df[["ramp_kit", "runtime_hyperopt"]].groupby(["ramp_kit"]).mean().sort_values(by="runtime_hyperopt")

In [ ]:
results_summary_df["kaggle_private_best"] = np.nan
results_summary_df["kaggle_private_prank_best"] = np.nan
results_summary_df["kaggle_private_percentage_improvement"] = np.nan
results_summary_df["kaggle_private_prank_improvement"] = np.nan
results_summary_df["kaggle_private_percentage_improvement_btb_over_lb"] = np.nan
results_summary_df["kaggle_private_prank_improvement_btb_over_lb"] = np.nan
for row_i, row in results_summary_df.iterrows():
    if row["run_finished"] == 1:
        kit_suffix = f"v{row['version']}_n{row['number']}"
        ramp_kit_dir = f"{row['ramp_kit']}_{kit_suffix}"
        metadata = json.load(open(kits_root/ramp_kit_dir / "data" / "metadata.json"))
        problem = rw.utils.assert_read_problem(ramp_kit_dir=kits_root/ramp_kit_dir)
        score_type = problem.score_types[-1]
        if score_type.is_lower_the_better:
            results_summary_df.loc[row_i, "kaggle_private_best"] = min(
                row["kaggle_private_lgbm"], row["kaggle_private_xgboost"], row["kaggle_private_catboost"])
            results_summary_df.loc[row_i, "kaggle_private_percentage_improvement"] =\
                200 * (results_summary_df.loc[row_i, "kaggle_private_best"] - row["kaggle_private_last_blend"]) /\
                (row["kaggle_private_best"] + row["kaggle_private_last_blend"])
            results_summary_df.loc[row_i, "kaggle_private_percentage_improvement_btb_over_lb"] =\
                200 * (row["kaggle_private_last_blend"] - row["kaggle_private_bagged_then_blended"]) /\
                (row["kaggle_private_bagged_then_blended"] + row["kaggle_private_last_blend"])
        else:
            results_summary_df.loc[row_i, "kaggle_private_best"] = max(
                row["kaggle_private_lgbm"], row["kaggle_private_xgboost"], row["kaggle_private_catboost"])
            results_summary_df.loc[row_i, "kaggle_private_percentage_improvement"] =\
                200 * (row["kaggle_private_last_blend"] - row["kaggle_private_best"]) /\
                (results_summary_df.loc[row_i, "kaggle_private_best"] + row["kaggle_private_last_blend"])
            results_summary_df.loc[row_i, "kaggle_private_percentage_improvement_btb_over_lb"] =\
                200 * (row["kaggle_private_bagged_then_blended"] - row["kaggle_private_last_blend"]) /\
                (row["kaggle_private_bagged_then_blended"] + row["kaggle_private_last_blend"])
        results_summary_df.loc[row_i, "kaggle_private_prank_best"] = max(
                row["kaggle_private_prank_lgbm"], row["kaggle_private_prank_xgboost"], row["kaggle_private_prank_catboost"])
        results_summary_df.loc[row_i, "kaggle_private_prank_improvement"] =\
            row["kaggle_private_prank_last_blend"] -  row["kaggle_private_prank_best"]
        results_summary_df.loc[row_i, "kaggle_private_prank_improvement_btb_over_lb"] =\
            row["kaggle_private_prank_bagged_then_blended"] -  row["kaggle_private_prank_last_blend"]


In [ ]:
p = figure(
    title=f"Relative improvement of blending over best Kaggle private scores", x_axis_label='Percentage',
    y_axis_label=f"Number of problems", width=800, height=400,
)

# Histogram
bins = np.linspace(-3, 3, 13)
hist, edges = np.histogram(results_summary_df["kaggle_private_percentage_improvement"], bins=bins)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:],
         fill_color="darkblue", line_color="white",
         legend_label=f"{len(results_summary_df[results_summary_df['run_finished'] == 1])} problems,"
                      f" mean = {round(results_summary_df['kaggle_private_percentage_improvement'].mean(), 1)}%" +\
                      f" median = {round(results_summary_df['kaggle_private_percentage_improvement'].median(), 1)}%")
show(p)

In [ ]:
p = figure(
    title=f"Relative improvement of blending over best Kaggle private rank", x_axis_label='Percentage point',
    y_axis_label=f"Number of problems", width=800, height=400,
)

# Histogram
bins = np.linspace(-12, 32, 45)
hist, edges = np.histogram(results_summary_df["kaggle_private_prank_improvement"], bins=bins)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:],
         fill_color="darkblue", line_color="white",
         legend_label=f"{len(results_summary_df[results_summary_df['run_finished'] == 1])} problems,"
                      f" mean = {round(results_summary_df['kaggle_private_prank_improvement'].mean(), 1)}%" +\
                      f" median = {round(results_summary_df['kaggle_private_prank_improvement'].median(), 1)}%")
show(p)

In [ ]:
p = figure(
    title=f"Relative improvement of bagged then blended over last blend Kaggle private scores", x_axis_label='Percentage',
    y_axis_label=f"Number of problems", width=800, height=400,
)

# Histogram
bins = np.linspace(-3, 3, 13)
hist, edges = np.histogram(results_summary_df["kaggle_private_percentage_improvement_btb_over_lb"], bins=bins)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:],
         fill_color="darkblue", line_color="white",
         legend_label=f"{len(results_summary_df[results_summary_df['run_finished'] == 1])} problems,"
                      f" mean = {round(results_summary_df['kaggle_private_percentage_improvement_btb_over_lb'].mean(), 1)}%" +\
                      f" median = {round(results_summary_df['kaggle_private_percentage_improvement_btb_over_lb'].median(), 1)}%")
show(p)

In [ ]:
p = figure(
    title=f"Relative improvement of bagged then blended over last blend Kaggle private rank", x_axis_label='Percentage point',
    y_axis_label=f"Number of problems", width=800, height=400,
)

# Histogram
bins = np.linspace(-12, 32, 45)
hist, edges = np.histogram(results_summary_df["kaggle_private_prank_improvement_btb_over_lb"], bins=bins)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:],
         fill_color="darkblue", line_color="white",
         legend_label=f"{len(results_summary_df[results_summary_df['run_finished'] == 1])} problems,"
                      f" mean = {round(results_summary_df['kaggle_private_prank_improvement_btb_over_lb'].mean(), 1)}%" +\
                      f" median = {round(results_summary_df['kaggle_private_prank_improvement_btb_over_lb'].median(), 1)}%")
show(p)